In [ ]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import cm
import seaborn as sns
sns.set(style='whitegrid')

In [ ]:
# do for width in 128 256 512
#     for num_exp in 8 4 2
#     do
#         for lr in 0.0078125 0.00390625 0.001953125 0.0009765625 0.00048828125 0.000244140625
#         do
#             for seed in 1

seed = 1  # Single seed
widths = [
    128,
]
num_exps = [8, 4, 2]  # Number of experts

lrs = [
    0.0078125,
    0.00390625,
    0.001953125,
    0.0009765625,
    0.00048828125,
    0.000244140625,
]

In [ ]:
# Create one large plot with all configurations
plt.figure(figsize=(12, 8))

# Color and marker helpers for different combinations
# Using a colormap for better visual distinction
cmap = plt.cm.tab10
colors = [cmap(i) for i in range(10)]
markers = ['o', 's', '^', 'D', 'v', '<', '>', 'p', 'h']
linestyles = ['-', '--', '-.']

# Track all configurations for legend
all_configs = []

for width_idx, width in enumerate(widths):
    for exp_idx, n_exp in enumerate(num_exps):
        num_act = int(n_exp / 2)
        losses = []
        lrs_to_plot = []
        
        for lr in lrs:
            # Construct the job name following the format from run.ps1
            job_name = f'width{width}_depth2_experts{n_exp}_active{num_act}_seed{seed}_lr{lr}'
            csv_path = os.path.join('mup_moe', 'out', job_name, 'log.csv')
            
            if os.path.exists(csv_path):
                try:
                    ckpt_df = pd.read_csv(csv_path)
                    if len(ckpt_df) > 50:  # Only include runs with sufficient iterations
                        # Use mean of last 20 iterations to get stable loss
                        losses.append(ckpt_df['train/loss'].tail(20).mean())
                        lrs_to_plot.append(lr)
                    else:
                        print(csv_path)
                except:
                    print(csv_path)
                    pass  # Skip corrupted files
        
        if len(losses) > 0:
            losses = np.array(losses)
            
            # Choose color and marker based on configuration
            config_idx = width_idx * len(num_exps) + exp_idx
            color = colors[config_idx % len(colors)]
            marker = markers[config_idx % len(markers)]
            linestyle = linestyles[exp_idx % len(linestyles)]
            
            # Plot (no error bars with single seed)
            line, = plt.plot(lrs_to_plot, losses, 
                   label=f'w={width}, exp={n_exp}, act={num_act}', 
                   marker=marker, 
                   color=color,
                   linestyle=linestyle,
                   linewidth=2,
                   markersize=6)
            all_configs.append(line)
            
            # Mark optimal learning rate with a star
            optimum_idx = np.argmin(losses)
            plt.plot(lrs_to_plot[optimum_idx], losses[optimum_idx], 
                   color=color, 
                   marker='*', markersize=12, 
                   markeredgecolor='black', markeredgewidth=1,
                   zorder=10)  # Ensure stars appear on top
        else:
            print(job_name)
plt.xscale('log', base=2)
plt.xlabel('Learning Rate', fontsize=14)
plt.ylabel('Training Loss', fontsize=14)
plt.title('μP Learning Rate Transfer for MOE Models (All Configurations)', fontsize=16)
plt.grid(True, alpha=0.3)

# Create a more compact legend
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10, ncol=1)

# Add annotations to distinguish width groups
ax = plt.gca()
y_min, y_max = ax.get_ylim()
text_y = y_max * 0.98

# Add text annotations for width groups
for width_idx, width in enumerate(widths):
    color_for_width = colors[(width_idx * len(num_exps)) % len(colors)]
    plt.text(0.02, 0.95 - width_idx * 0.04, f'Width {width}', 
             transform=ax.transAxes, fontsize=10, 
             bbox=dict(boxstyle="round,pad=0.3", facecolor=color_for_width, alpha=0.3))

plt.tight_layout()
plt.savefig('mup_moe_lr_transfer_merged.png', dpi=300, bbox_inches='tight')
plt.show()

# Print optimal learning rates for each configuration
print("\nOptimal Learning Rates:")
print("=" * 50)
for n_exp in num_exps:
    num_act = int(n_exp / 2)
    print(f"\nExperts = {n_exp}:")
    for width in widths:
        losses = []
        lrs_list = []
        for lr in lrs:
            job_name = f'width{width}_depth2_experts{n_exp}_active{num_act}_seed{seed}_lr{lr}'
            csv_path = os.path.join('mup_moe', 'out', job_name, 'log.csv')
            if os.path.exists(csv_path):
                try:
                    ckpt_df = pd.read_csv(csv_path)
                    if len(ckpt_df) > 50:
                        losses.append(ckpt_df['train/loss'].tail(20).mean())
                        lrs_list.append(lr)
                except:
                    print(csv_path)
                    pass
        
        if len(losses) > 0:
            optimum_idx = np.argmin(losses)
            print(f"  Width {width}: LR = {lrs_list[optimum_idx]:.6f}, Loss = {losses[optimum_idx]:.4f}")